# Очистка данных: Spend

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import help_130625_dam as h

DATA_PATH = os.path.join('..', 'Sources', 'Spend (Done).xlsx')
OUT_PATH  = os.path.join('..', 'data', 'cleaned', 'spend_clean.pkl')

pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True, precision=2)

## Загрузка и первичный осмотр

In [2]:
df = pd.read_excel(DATA_PATH)

# Переименование столбцов в snake_case
df.columns = [h.to_snake(c) for c in df.columns]

print(f'Форма: {df.shape}')

n_before = df.shape[0]
h.descr_df(df, include='all', show_sample_rows=True)

Форма: (20779, 8)


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,date,datetime64[us],20779,0,355,2023-07-03 00:00:00,2023-07-03 00:00:00,2023-07-03 00:00:00,NaN,NaN,NaN,NaN
1,source,str,20779,0,14,Google Ads,Google Ads,Facebook Ads,NaN,NaN,NaN,NaN
2,campaign,str,14785,5994,51,gen_analyst_DE,performancemax_eng_DE,NaN,NaN,NaN,NaN,NaN
3,impressions,int64,20779,0,4003,6,4,0,0.00,2458.20,63.00,431445.00
4,spend,float64,20779,0,2859,0.00,0.01,0.00,0.00,7.20,0.58,774.00
5,clicks,int64,20779,0,552,0,1,0,0.00,23.99,1.00,2415.00
6,adgroup,str,13951,6828,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ad,str,13951,6828,176,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Пропущенные значения
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
display(
    pd.DataFrame({'Пропуски': missing, '% пропусков': missing_pct})
    .query('Пропуски > 0')
)
print('Строк без пропусков:', df.dropna().shape[0])

,Пропуски,% пропусков
campaign,5994,28.85
adgroup,6828,32.86
ad,6828,32.86


Строк без пропусков: 13874


In [4]:
full_dupes = df.duplicated().sum()
print(f'Полных дубликатов: {full_dupes}')

# У Spend нет Id — дубликат это полное совпадение всех полей
if full_dupes > 0:
    print('Пример дубликатов:')
    dupes_df = df[df.duplicated(keep=False)].sort_values('date')
    display(dupes_df.head(10))
    
    print("\nИтоги по числовым полям в дубликатах (сумма):")
    # Вычисляем суммы только для строк, которые будут удалены (keep='first' оставит одну, остальные в расчет)
    removed_dupes = df[df.duplicated(keep='first')]
    summary_dupes = removed_dupes[['impressions', 'spend', 'clicks']].sum()
    display(summary_dupes.to_frame('Сумма удаляемых данных'))
    
    df = df.drop_duplicates().reset_index(drop=True)

print(f'Строк до: {n_before}  →  после: {len(df)}  (удалено: {n_before - len(df)})')

Полных дубликатов: 917
Пример дубликатов:


,date,source,campaign,impressions,spend,clicks,adgroup,ad
753,2023-07-23,Bloggers,NaN,0,0.00,0,NaN,NaN
755,2023-07-23,Bloggers,NaN,0,0.00,0,NaN,NaN
768,2023-07-24,Bloggers,NaN,0,0.00,0,NaN,NaN
789,2023-07-24,Bloggers,NaN,0,0.00,0,NaN,NaN
841,2023-07-25,Bloggers,NaN,0,0.00,0,NaN,NaN
844,2023-07-25,Bloggers,NaN,0,0.00,0,NaN,NaN
895,2023-07-26,Bloggers,NaN,0,0.00,0,NaN,NaN
899,2023-07-26,Bloggers,NaN,0,0.00,0,NaN,NaN
950,2023-07-27,Bloggers,NaN,0,0.00,0,NaN,NaN
958,2023-07-27,Bloggers,NaN,0,0.00,0,NaN,NaN



Итоги по числовым полям в дубликатах (сумма):


,Сумма удаляемых данных
impressions,0.00
spend,0.00
clicks,46.00


Строк до: 20779  →  после: 19862  (удалено: 917)


## Типы данных: дата

In [5]:
DATE_COLS = ['date']

for col in DATE_COLS:
    # Явно указываем формат YYYY-MM-DD
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

# Проверяем результат
print('Типы после парсинга:')
print(df[DATE_COLS].dtypes)
print()

# Считаем NaT = не распарсились
for col in DATE_COLS:
    nat_count = df[col].isna().sum()
    print(f'{col}: NaT = {nat_count} ({nat_count/len(df)*100:.2f}%)')

Типы после парсинга:
date    datetime64[us]
dtype: object

date: NaT = 0 (0.00%)


## Числовые поля

In [6]:
# Проверяем наличие "грязных" значений в числовых полях
NUM_COLS = ['impressions', 'spend', 'clicks']

for col in NUM_COLS:
    print(f'--- {col} ---')
    print(f'  Тип: {df[col].dtype}')
    print(f'  Мин: {df[col].min()}, Макс: {df[col].max()}')
    neg = (df[col] < 0).sum()
    print(f'  Отрицательных значений: {neg}')

--- impressions ---
  Тип: int64
  Мин: 0, Макс: 431445
  Отрицательных значений: 0
--- spend ---
  Тип: float64
  Мин: 0.0, Макс: 774.0
  Отрицательных значений: 0
--- clicks ---
  Тип: int64
  Мин: 0, Макс: 2415
  Отрицательных значений: 0


## Пропущенные значения

Этот датасет это единственный источник значений campaign, adgroup, ad, поэтому дозаполнить их не 
получиться. Пропущенные значения заменяем на Unknown и меняем тип на category

In [7]:
# campaign, adgroup, ad — ~30% пропусков, заполняем 'Unknown'
FILL_UNKNOWN = ['campaign', 'adgroup', 'ad']

for col in FILL_UNKNOWN:
    n_miss = df[col].isna().sum()
    df[col] = df[col].fillna('Unknown')
    print(f'{col}: заполнено {n_miss} пропусков → "Unknown"')

# Проверка
print('\nПропуски после заполнения:')
missing_after = df.isnull().sum()
display(
    pd.DataFrame({'Пропуски': missing_after, '% пропусков': (missing_after / len(df) * 100).round(2)})
    .query('Пропуски > 0')
)

CAT_COLS = ['source', 'campaign', 'adgroup', 'ad']

for col in CAT_COLS:
    # Нормализация: убираем лишние пробелы по краям
    df[col] = df[col].str.strip()
    df[col] = df[col].astype('category')
    print(f'{col}: {df[col].nunique()} уникальных значений')

campaign: заполнено 5077 пропусков → "Unknown"
adgroup: заполнено 5911 пропусков → "Unknown"
ad: заполнено 5911 пропусков → "Unknown"

Пропуски после заполнения:


,Пропуски,% пропусков


source: 14 уникальных значений
campaign: 52 уникальных значений
adgroup: 25 уникальных значений
ad: 177 уникальных значений


## Итоги

In [8]:
print(f'Итоговая форма датасета: {df.shape}')

h.descr_df(df, include='all', show_sample_rows=True)

Итоговая форма датасета: (19862, 8)


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,date,datetime64[us],19862,0,355,2023-07-03 00:00:00,2023-07-03 00:00:00,2023-07-03 00:00:00,NaN,NaN,NaN,NaN
1,source,category,19862,0,14,Google Ads,Google Ads,Facebook Ads,NaN,NaN,NaN,NaN
2,campaign,category,19862,0,52,gen_analyst_DE,performancemax_eng_DE,Unknown,NaN,NaN,NaN,NaN
3,impressions,int64,19862,0,4003,6,4,0,0.00,2571.70,82.00,431445.00
4,spend,float64,19862,0,2859,0.00,0.01,0.00,0.00,7.53,0.74,774.00
5,clicks,int64,19862,0,552,0,1,0,0.00,25.10,2.00,2415.00
6,adgroup,category,19862,0,25,Unknown,Unknown,Unknown,NaN,NaN,NaN,NaN
7,ad,category,19862,0,177,Unknown,Unknown,Unknown,NaN,NaN,NaN,NaN


## Сохранение

In [ ]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_pickle(OUT_PATH)
df.to_excel(OUT_PATH.replace('.pkl', '.xlsx'))

# Итоги
summary_data = {
    'Метрика': [
        'Строк исходно',
        'Строк после очистки',
        'Удалено дубликатов',
        'Диапазон дат',
        'Каналов (source)',
        'Итого Spend, €',
        'Пропуски после заполнения'
    ],
    'Значение': [
        n_before,
        len(df),
        n_before - len(df),
        f'{df["date"].min().date()} → {df["date"].max().date()}',
        df['source'].nunique(),
        f'{df["spend"].sum():,.2f}',
        df.isnull().sum().sum()
    ]
}

print(f'Сохранено: {OUT_PATH}')
display(pd.DataFrame(summary_data))


Сохранено: ..\data\cleaned\spend_clean.pkl

,Метрика,Значение
0,Строк исходно,20779
1,Строк после очистки,19862
2,Удалено дубликатов,917
3,Диапазон дат,2023-07-03 → 2024-06-21
4,Каналов (source),14
5,"Итого Spend, €","149,523.45"
6,Пропуски после заполнения,0


## Статистика

In [10]:
# Сводная статистика для числовых полей
# Рассчитываем среднее, медиану, моду и диапазон
num_stats = df[NUM_COLS].agg([
    'mean', 
    'median', 
    lambda x: x.mode()[0], 
    lambda x: x.max() - x.min()
]).T
num_stats.columns = ['Mean', 'Median', 'Mode', 'Range']

print("Сводная статистика числовых полей:")
display(num_stats)

# Анализ категориальных полей

print("\nРаспределение по источникам (Top 10):")
display(df['source'].value_counts().head(10).to_frame('Записей'))

print("\nРаспределение по кампаниям (Top 10):")
display(df['campaign'].value_counts().head(10).to_frame('Записей'))

Сводная статистика числовых полей:


,Mean,Median,Mode,Range
impressions,2571.70,82.00,0.00,431445.00
spend,7.53,0.74,0.00,774.00
clicks,25.10,2.00,0.00,2415.00



Распределение по источникам (Top 10):


,Записей
source,
Facebook Ads,9569
Tiktok Ads,2985
Youtube Ads,1784
Google Ads,1266
Telegram posts,836
Webinar,766
Bloggers,632
SMM,571
Organic,514



Распределение по кампаниям (Top 10):


,Записей
campaign,
Unknown,5077
12.07.2023wide_DE,2073
02.07.23wide_DE,1685
04.07.23recentlymoved_DE,1398
youtube_shorts_DE,1223
07.07.23LAL_DE,1181
03.07.23women,1171
12.09.23interests_Uxui_DE,1143
15.07.23b_DE,529


## Описание датасета

**Источник:** `Spend.xlsx` — данные рекламных кабинетов  
**Назначение:** учет маркетинговых затрат для расчета ROI/ROAS и анализа эффективности каналов привлечения

| Столбец | Тип | Описание |
|---|---|---|
| `date` | `datetime` | Дата расхода |
| `source` | `category` | Рекламный канал/источник (Google, Facebook и др.) |
| `campaign` | `category` | Название рекламной кампании |
| `impressions` | `int64` | Количество показов объявлений |
| `spend` | `float64` | Сумма фактических затрат в валюте кабинета |
| `clicks` | `int64` | Количество переходов (кликов) |
| `adgroup` | `category` | Группа объявлений |
| `ad` | `category` | Конкретное объявление/креатив |

**Объём:** 19 862 записей (после очистки), 8 столбцов  
**Ключевые связи:**
- `date` + `source` → агрегация для сопоставления с результатами продаж в `06_analytics`

Удалено 917 дубликатов, в которых практически не было данных. Типы данных преобразованы для дальнейшего анализа.